# LABORATORIO 8: FASES Y DATOS SEMIESTRUCTURADOS

## Crear un objeto de fase(stage) de Snowflake

👉 En esta lección, crearemos una fase que señala un contenedor **S3** (Amazon Simple Storage Service) que el equipo de Servicios de Educación creó para que usted use. 

Para comenzar, reunamos la **información de contexto** que usaremos durante este laboratorio. 

- Haga clic en el botón **Start** para activar este cuaderno.

- Ejecute la siguiente celda Python.


#### :warning: Debe volver a ejecutar la celda a continuación siempre que se inicie una nueva sesión para este cuaderno para configurar “variables” para usar en celdas posteriores. :advertencia:


In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('Su información de contexto actual:')
print('---------------------------------')
print(session)
print('Su usuario actual es' + user)

### Creación de fases 🥋

1. En el navegador de objetos de Snowsight, vaya a **Catalog** > **Database Explorer** y seleccione la base de datos **(animal)_UTIL_DB** que creó en un laboratorio anterior.
1. Luego, seleccione el esquema con el nombre **PUBLIC**.
1. Haga clic en el botón azul **Create** en la parte superior derecha.
1. Seleccione **Stage** > **External Stage** > **Amazon S3**.

![Opción para crear fase (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_create_stage_new_1.png)

Aparecerá un cuadro de diálogo para crear una fase.

![Cuadro de diálogo para crear una fase (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_create_stage_dialog_1.png)

1. En el cuadro de texto **Stage Name**, ingrese: `like_a_window_into_an_s3_bucket`.
1. Ingrese el siguiente nombre en el cuadro de texto **URL**: `s3://uni-lab-files`.
1. Asegúrese de que la opción **Directory table** esté seleccionada.
1. Haga clic en el botón azul **Create** en la esquina inferior derecha.
1. La siguiente pantalla puede solicitar el nombre de un almacén virtual de Snowflake para usar; elija el almacén con el nombre **(animal)_WH**.


### El navegador de objetos muestra los archivos disponibles en esta nueva fase que creó 📓

![Archivos de la fase (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_new_stage_files.png)


💡 **Consejo**: Estos archivos y carpetas están en un contenedor AWS S3 que es propiedad y está administrado por el equipo de Servicios de Educación de Snowflake que le provee este curso. Usted puede ver aquí la lista de archivos porque creó una fase, la que actúa como una ventana que le permite ver y acceder a los archivos en ese contenedor. Nuestro contenedor es público, pero los contenedores que crea su compañía probablemente requerirán credenciales.


### ¿Es una fase o no es una fase? 📓 

Un aspecto potencialmente confuso sobre las fases de Snowflake es que el objeto de fase que creó no es una ubicación. La ubicación (el contenedor S3 que retiene los archivos) ya existía. Entonces, ¿qué acaba de crear? 

Pues bien, usted creó algo que le comunica a Snowflake cierta información sobre una ubicación en la que ya hay algunos archivos preconfigurados(staged). No creó la ubicación real de fase; creó algo que se asemeja más a una ventana en esa ubicación de fase. Su objeto de fase de Snowflake es casi como un formato de archivo en el que se retiene información de configuración que facilita la carga de archivos.

A veces, cuando definimos una fase de Snowflake, también proporcionamos credenciales de acceso, pero en este caso, no lo hicimos. Nuestra fase es solo un objeto al que asignamos un nombre que señala a un contenedor S3 en donde ya hay algunos archivos preconfigurados.  


## Usar el comando `LIST` desde una celda SQL 🥋 

### Usar el comando `LIST` para ver los archivos en su nueva fase 🥋 

El comando `LIST` devuelve una lista de archivos preconfigurados (es decir, cargados desde un sistema de archivos local o descargados de una tabla) en una fase de Snowflake. 
- Este comando también puede abreviarse como `LS`.
- Se refiere a un objeto de Fase (stage) en Snowflake por su nombre, con un prefijo con un carácter de arroba `@`.

Pruebe el siguiente comando `LIST` por su cuenta.


In [ ]:
USE SCHEMA {{user}}_UTIL_DB.PUBLIC;

LIST @like_a_window_into_an_s3_bucket;

### Convenciones para la asignación de nombres de los objetos de Snowflake 📓

Tenga en cuenta que Snowflake no distingue entre mayúsculas y minúsculas. Ingresamos el nombre de nuestro objeto de fase en minúsculas. Snowflake siempre supone que realmente tiene intenciones de escribir todo en **MAYÚSCULAS**, por lo que lo convierte por usted. Debido a esto, puede escribir en minúsculas o una combinación de mayúsculas y minúsculas al crear o consultar objetos, y Snowflake lo convertirá a todo mayúsculas en segundo plano.

(A menos que use comillas al crear cosas, luego tendrá que usar comillas siempre después de esto para lidiar con el objeto).

Entonces, al ejecutar comandos en la fase, usar mayúsculas o minúsculas funcionará. No obstante, S3 es muy particular, de modo que debe ser muy disciplinado una vez que complete el nombre del objeto de fase. Debe usar la ortografía exacta, con las mayúsculas y minúsculas correctas incluso para la extensión del archivo.  

![Archivos de la fase (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_list_command_2.png)


### :mag_right: Comprobación 8 (OB08) 🔎

- ¿Creó una fase externa con el nombre `like_a_window_into_an_s3_bucket`?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB08', '{{user}}');

## Use la instrucción `COPY INTO` para cargar datos 🥋 

### Cree una tabla para tipos de suelo 🥋 

Asegúrese de crearla en la base de datos **(animal)_GARDEN_PLANTS** del esquema **VEGGIES**. Para hacerlo, debe alterar la LÍNEA 2 en la siguiente celda SQL, reemplazando los caracteres hash `('#')` por el nombre de esquema correcto antes de ejecutar la celda.


In [ ]:
-- reemplace los caracteres de hash ('#') en la siguiente línea
USE SCHEMA {{user}}_GARDEN_PLANTS.VEGGIES;

CREATE OR REPLACE TABLE vegetable_details_soil_type
( plant_name VARCHAR(25)
 ,soil_type NUMBER(1,0)
);

### Cargue un archivo del contenedor S3 a la tabla nueva 📓

Anteriormente usó la pantalla **Load Data** de Snowsight para copiar datos de un archivo preconfigurado a una tabla. Al usar esta estrategia “impulsada por el asistente”, Snowflake genera y ejecuta código en segundo plano para llevar a cabo esta acción. Para esta carga de datos, exploraremos la estrategia programática.

Usaremos una instrucción `COPY INTO` desde dentro de una celda SQL del cuaderno.

Para usar la instrucción `COPY INTO`, es mejor tener cuatro cosas:

1. Una tabla 

1. Un objeto de fase

1. Un archivo

1. Un formato de archivo (opcional)

El **formato de archivo** es opcional porque hay una alternativa, pero es un proceso más claro si tiene uno. Como se mencionó anteriormente, el formato de archivo es un objeto que proporciona a Snowflake instrucciones sobre cómo manipular los datos que se cargan desde una fase. En el siguiente ejemplo, proporcionaremos estas “instrucciones” en línea en lugar de definir un formato de archivo.


### Una instrucción `COPY INTO` que puede ejecutar 🥋 


In [ ]:
COPY INTO vegetable_details_soil_type
FROM @{{user}}_util_db.public.like_a_window_into_an_s3_bucket
FILES = ( 'VEG_NAME_TO_SOIL_TYPE_PIPE.txt')
FILE_FORMAT = (
    TYPE=csv
    FIELD_DELIMITER = '|'
    SKIP_HEADER=1
);

### :mag_right: Comprobación 9 (OB09) 🔎

- ¿Tiene 42 filas cargadas en la tabla **vegetable_details_soil_type** del esquema **(animal)_garden_plants.veggies**?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB09', '{{user}}');

## Consejos y trucos para cargar datos 📓

- Todos los archivos sin formato se cargan usando formatos de archivo que tienen un tipo de CSV (valores separados por comas). Entonces, use `TYPE = CSV` para cualquier archivo sin formato (TSV, delimitado por líneas, .txt y demás).

- La propiedad **FIELD_DELIMITER** es muy importante. Debe coincidir con el separador de columnas real que se usa en el archivo. 

- El asistente para la carga de datos puede ayudarle a escribir su formato de archivo (con nombre o en línea). Simplemente elija la configuración que necesita en las listas desplegables y luego haga clic en el enlace Show SQL para acceder al código SQL para estos ajustes. 


## Ejercicio de desafío: Crear una tabla de búsqueda de tipos de suelo 🎯 

En este ejercicio de desafío, deberá crear una nueva tabla con el nombre **lu_soil_type** y cargarla con datos de un archivo suministrado mediante uno de los siguientes métodos:
- Uso del asistente para **cargar datos** **O**
- Redacción de su propia instrucción `COPY INTO` y su ejecución

:warning: Solo se suministrarán instrucciones generales para este ejercicio y deberá “descifrar los pasos” con base en el material que ya hemos abordado. :warning:

Primero, cree la tabla. Asegúrese de crearla en la base de datos **(animal)_GARDEN_PLANTS** del esquema **VEGGIES**. 


In [ ]:
USE SCHEMA {{user}}_GARDEN_PLANTS.VEGGIES;

CREATE OR REPLACE TABLE lu_soil_type(
    soil_type_id NUMBER,	
    soil_type VARCHAR(15),
    soil_description VARCHAR(75)
);

### Descargue el archivo de datos de origen 🎯 

Ejecute la siguiente celda de código Python y haga clic en el enlace generado para descargar el archivo **LU_SOIL_TYPE.tsv**.


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'LU_SOIL_TYPE.tsv')")
collected_data = snowpark_df.collect()
st.write('Haga clic en el siguiente enlace para descargar el archivo:')
st.write(collected_data[0][0])

### Cargue filas de tabla del archivo descargado 🎯

El archivo **LU_SOIL_TYPE.tsv** comparte muchas de las propiedades de formato de archivo con el archivo que cargó antes, **VEG_NAME_TO_SOIL_TYPE_PIPE.txt**, con una importante excepción. Vamos a resumirlas:
- **TYPE**: el archivo tiene una extensión **.tsv** (por lo que debería poder determinar su [tipo](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table#type-csv)).
- **SKIP_HEADER**: el archivo tiene una fila de encabezado.
- **FIELD_DELIMITER**: el archivo **NO ESTÁ** delimitado por líneas (`'|'`) está delimitado por **TAB**, que está representado por los siguientes caracteres: `'\t'`.


#### Opción 1:
- Use el asistente para **cargar datos** en Snowsight.

#### U

#### Opción 2:
- Con las notas anteriores, modifique y ejecute el siguiente fragmento `COPY INTO` SQL.
- Suministre el nombre de la tabla objetivo (línea 1).
- Suministre las opciones de formato de archivo (líneas de la 5 a la 7).


In [ ]:
COPY INTO LU_SOIL_TYPE
FROM @{{user}}_util_db.public.like_a_window_into_an_s3_bucket
FILES = ( 'LU_SOIL_TYPE.tsv')
FILE_FORMAT = (
    TYPE=csv
    FIELD_DELIMITER = '\t'
    SKIP_HEADER=1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
);

### :mag_right: Comprobación 10 (OB10) 🔎

- ¿Tiene ocho filas cargadas en la tabla **lu_soil_type** del esquema **(animal)_garden_plants.veggies**?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB10', '{{user}}');

## Trabaje con datos semiestructurados 📓

Los datos semiestructurados son datos que no cumplen con los estándares de datos estructurados tradicionales, pero incluyen etiquetas u otros tipos de marcado que identifican entidades individuales y diferentes dentro de los datos. Dos atributos clave que distinguen a los datos semiestructurados de los datos estructurados son las estructuras de datos anidadas y la falta de un esquema fijo:

- Los datos semiestructurados no requieren una definición previa de un esquema y pueden evolucionar de manera constante (se pueden agregar nuevos atributos en cualquier momento).
- A diferencia de los datos estructurados, que representa los datos como una tabla sin procesar, los datos semiestructurados pueden incluir jerarquías de N niveles de información anidada.

Aquí se incluye un ejemplo de datos JSON, que es uno de los tipos semiestructurados admitidos por Snowflake.

![Datos de la tabla de detalles de verduras (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_json_data_extract.png)

### El tipo de datos `VARIANT`.

Snowflake ofrece el tipo de dato [`VARIANT`](https://docs.snowflake.com/en/sql-reference/data-types-semistructured#label-data-type-variant) para admitir el almacenamiento de datos semiestructurados. Este es un tipo de dato que puede contener un valor de cualquier otro tipo de dato, incluidos [`ARRAY` y ](https://docs.snowflake.com/en/sql-reference/data-types-semistructured#array)[`OBJECT`](https://docs.snowflake.com/en/sql-reference/data-types-semistructured#object) (que también se utilizan a menudo junto con datos semiestructurados). Con este tipo de datos, podemos preservar el formato jerárquico/anidado de los datos semiestructurados al ingerirlos en Snowflake.


### Crear una tabla que incluya una columna `VARIANT` 🥋

Cree una tabla con el nombre **VEGETABLE_DETAILS_PLANT_HEIGHT** en el esquema **VEGGIES** de la base de datos **(animal)_GARDEN_PLANTS**. Esto contiene una columna individual del tipo de datos `VARIANT`. 


In [ ]:
CREATE OR REPLACE TABLE {{user}}_GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS_PLANT_HEIGHT (
	record VARIANT
);

### Descargue el archivo de datos de origen JSON 🥋

Ejecute la siguiente celda de código Python y haga clic en el enlace generado para descargar el archivo JSON **veg_plant_height.json**.

Según su navegador, esto puede abrir de manera directa en una nueva pestaña/ventana o descargarse. Si se descarga, abra el archivo en un editor de texto de su sistema local para revisar su estructura.

💡 **Consejo**: Descargue este archivo para revisar su estructura y contenido. Al cargar los datos desde este archivo, usaremos el archivo que ya se encuentra en la fase(stage).


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'veg_plant_height.json')")
collected_data = snowpark_df.collect()
st.write('Haga clic en el siguiente enlace para descargar el archivo:')
st.write(collected_data[0][0])

### Se puede ejecutar una instrucción `COPY INTO` para cargar datos JSON 🥋 

Ahora ejecute la siguiente instrucción para cargar estos datos JSON en su nueva tabla.

- ¿Qué ha cambiado en las siguientes especificaciones de **formato de archivo** para adaptar la carga de datos semiestructurados en lugar de estructurados?


In [ ]:
COPY INTO vegetable_details_plant_height
FROM @common_db.resources.course_files/veg_plant_height.json
FILE_FORMAT = (TYPE = 'JSON');

### Consulte la tabla cargada de datos JSON 🥋

Snowflake tiene operadores y funciones especiales para consultar datos jerárquicos complejos almacenados en `VARIANT`. Los presentaremos en breve, pero por ahora, revisemos una fila individual de datos de la tabla **vegetable_details_plant_height** mediante la sintaxis SQL regular del tipo que podríamos usar con conjuntos de datos estructurados.


In [ ]:
SELECT *
FROM vegetable_details_plant_height
LIMIT 1;

### Consulte datos semiestructurados 

Lo que observó en la consulta anterior es que la columna `VARIANT` con el nombre **RECORD** incluye un conjunto de cuatro valores de par de claves. Esto está bien, con datos retenidos en el formato en el que se ingirieron. No obstante, probablemente haya momentos en que querrá aislar elementos individuales para procesar y usar en un formato estructurado para informes. 

Aquí se incluyen algunas pautas detalladas para **atravesar** datos semiestructurados en Snowflake. 
- Inserte dos puntos `:` entre el nombre de una columna `VARIANT` y cualquier elemento de primer nivel.
    - <column>:<level1_element>
- Use notación por puntos para acceder a elementos posteriores anidados en la ruta jerárquica de un objeto JSON: 
    - <column>:<level1_element>.<level2_element>.<level3_element>
- El **nombre de la columna** no distingue entre mayúsculas y minúsculas, pero los nombres de los elementos **sí distinguen** entre mayúsculas y minúsculas.
- Como opción, puede encerrar los nombres de elementos entre comillas dobles.

Pruebe este simple ejemplo para **aplanar** (que implica representar datos estructurados) las filas en la tabla **vegetable_details_plant_height**.


In [ ]:
//Returns the data in a way that makes it look like a normalized table
SELECT 
    record:PLANT_NAME::STRING AS plant_name,
    record:UOM AS uom, -- sin casting
    record:LOW_END_OF_RANGE::INTEGER AS low_end_of_range,
    record:HIGH_END_OF_RANGE::INTEGER AS high_end_of_range
FROM vegetable_details_plant_height;

### Casteo (casting) de valores `VARIANT` 📓

Lo que posiblemente haya observado cuando ejecutó la consulta anterior fue que los valores de la columna **UOM** no se **castearon** (convirtieron a otros tipos de datos) como los demás, de modo que se ven algo diferentes. Se encerraron entre comillas dobles, como se ve a continuación: “F”.

Esto no significa que los valores de las columnas sean `VARCHAR` o `STRING` (su sinónimo); sino que indica que todavía es un valor `VARIANT`. Los valores `VARIANT` no son cadenas; en cambio, los valores `VARIANT` incluyen cadenas.

En Snowflake SQL, puede castear tipos de datos de uno a otro de las siguientes formas:
- Ejecute la función [`CAST()`](https://docs.snowflake.com/en/sql-reference/functions/cast).
- Ejecute el operador `::` como una sintaxis alternativa (por ejemplo, **record:PLANT_NAME::STRING**)


## Ejercicio de desafío: Crear una vista para mostrar datos semiestructurados 🎯 

En este ejercicio de desafío, va a crear un nuevo objeto [Vista](https://docs.snowflake.com/en/user-guide/views-introduction). Una Vista permite acceder al resultado de una consulta como si fuera una tabla. El objetivo es presentar los datos en **vegetable_details_plant_height** de un modo normalizado.

1. Modifique SQL según la complejidad en la celda que se encuentra debajo; las tres líneas que requieren cambios se han identificado con `-- ***`.
1. **CASTEE** la columna **UOM** en `VARCHAR`.
1. **CAMBIE** el orden de **LOW_END_OF_RANGE** y las columnas **HIGH_END_OF_RANGE**.


In [ ]:
-- modifique el siguiente código según las instrucciones anteriores, luego ejecútelo para crear el objeto
CREATE OR REPLACE VIEW vegetable_details_plant_height_vw AS 
SELECT 
    record:PLANT_NAME::STRING AS plant_name,
    record:UOM::VARCHAR AS uom,
    record:HIGH_END_OF_RANGE::INTEGER AS high_end_of_range,
    record:LOW_END_OF_RANGE::INTEGER AS low_end_of_range
FROM vegetable_details_plant_height;

In [ ]:
-- luego ejecute esta consulta en su nueva vista para confirmar la salida"
SELECT *
FROM vegetable_details_plant_height_vw;

### :mag_right: Comprobación 11 (OB11) 🔎

- ¿Creó una vista con el nombre **vegetable_details_plant_height_vw** en el esquema **VEGGIES** de la base de datos **(animal)_GARDEN_PLANTS**?
- ¿Casteó **UOM** en una columna `VARCHAR` (texto) con la columna **HIGH_END_OF_RANGE** en la tercera posición y **LOW_END_OF_RANGE** en la cuarta?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB11', '{{user}}');

## Ponga a prueba sus conocimientos :mag_right:

Compruebe sus conocimientos con las preguntas interactivas que aparecen a continuación. Cada celda `RUN_THIS_QUIZ_QUESTION_` tiene un widget Streamlit que presenta una pregunta de opción múltiple sobre la funcionalidad de Snowflake.  

**Instrucciones:**  
1. Pase el cursor sobre la celda del cuaderno para revelar controles adicionales.
1. Haga clic en el botón ▶️ **Reproducir** en el lado derecho de cada celda del cuestionario para ejecutarla.  
1. Seleccione su respuesta entre las opciones que se proporcionan.  
1. Revise los comentarios antes de continuar. 

💡 **Nota:** Puede expandir la celda para ver el código si tiene curiosidad, pero no es obligatorio. Estos cuestionarios no son obligatorios: están disponibles para desafiar su transferencia de conocimientos y proporcionarle la oportunidad de practicar.  


In [ ]:
st.divider()
question = "¿Por qué crees que le dimos el nombre LIKE_A_WINDOW_INTO_AN_S3_BUCKET a nuestro objeto de escenario de Snowflake?"
options = ["Seleccione una opción a continuación...",
           "A) Porque es corto, por lo que es fácil de escribir", 
           "B) Porque las fases son fáciles de romper, al igual que las ventanas pueden serlo a veces", 
           "C) Porque \"stage\" (fase) suena como un lugar, y en este caso, el lugar es el bucket de S3, no el objeto Snowflake que creamos"]                                 

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...": # esta opción es una solución alternativa hasta que Streamlit se actualice a > 1.26.0 para que podamos usar index=None
        ''
    else:
        answer = '44c17676896133f19e8a3384d82ef9df'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "MAYÚSCULAS, minúsculas, miXeD CaSe, ¿A quién le importa? ¿Cuáles de las siguientes afirmaciones son verdaderas sobre la sensibilidad a mayúsculas y minúsculas de Snowflake?"
options = ["Seleccione una opción a continuación...",
           "A) Snowflake asume que siempre quisiste escribir en MAYÚSCULAS, a menos que pongas algo entre comillas dobles", 
           "B) Si creas una tabla llamada HAPPY, Snowflake la nombrará en realidad happy", 
           "C) Si creas una tabla llamada \"hAppY\", cada vez que la consultes, debes ponerla entre comillas simples"]                                 

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Selecciona una opción a continuación...": # esta opción es una solución alternativa hasta que Streamlit se actualice a > 1.26.0 para que podamos usar index=None
        ''
    else:
        answer = '2c78194d16f97b5605cef95232153827'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## Siguientes pasos

Si completó los pasos del laboratorio y respondió correctamente las preguntas de la **Prueba de conocimientos**, continúe al siguiente cuaderno cuando se lo indique el instructor de Snowflake.
